In [ ]:

from datetime import datetime, timezone
import json
import uuid

# Synthetic maintenance report
report = {
    "report_id": "MRO-TEST-001",
    "aircraft": "SIMULATED-AIRCRAFT",
    "description": "Hydraulic fluid leakage reported during a fictional inspection.",
    "status": "NEW"
}

# Deterministic demonstration workflow (no LLM yet)
def analyze_report(report):
    request_id = str(uuid.uuid4())

    description = report["description"].lower()

    if "leak" in description:
        category = "FLUID_LEAK_REPORT"
    else:
        category = "REQUIRES_MANUAL_CLASSIFICATION"

    result = {
        "request_id": request_id,
        "report_id": report["report_id"],
        "category": category,
        "recommendation": "Refer to authorized human reviewer",
        "approval_status": "PENDING",
        "operational_authorization": False,
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    return result

result = analyze_report(report)

print(json.dumps(result, indent=2))


{
  "request_id": "4e2fbfd5-71c4-4f7e-8bc5-dcc927cb0d16",
  "report_id": "MRO-TEST-001",
  "category": "FLUID_LEAK_REPORT",
  "recommendation": "Refer to authorized human reviewer",
  "approval_status": "PENDING",
  "operational_authorization": false,
  "timestamp": "2026-10-15T06:19:08.870438+00:00"
}


In [ ]:

# CTRL-03 – Role-Based Access Control (RBAC)
# Simulated authorization test

ROLE_PERMISSIONS = {
    "maintenance_planner": {"analyze_report"},
    "quality_specialist": {"analyze_report"},
    "auditor": {"view_audit_log"},
    "guest": set()
}

def authorize(role, action):
    allowed = action in ROLE_PERMISSIONS.get(role, set())

    return {
        "role": role,
        "action": action,
        "access_granted": allowed,
        "decision": "ALLOW" if allowed else "DENY"
    }

# TEST-03: Authorized and unauthorized access
test_cases = [
    ("maintenance_planner", "analyze_report", True),
    ("guest", "analyze_report", False),
    ("auditor", "analyze_report", False),
    ("unknown_role", "analyze_report", False)
]

for role, action, expected in test_cases:
    result = authorize(role, action)
    actual = result["access_granted"]
    status = "PASS" if actual == expected else "FAIL"

    print(f"{status} | {role} | {result['decision']}")

    assert actual == expected, f"Unexpected result for {role}"


PASS | maintenance_planner | ALLOW
PASS | guest | DENY
PASS | auditor | DENY
PASS | unknown_role | DENY


In [ ]:

# CTRL-03 – Access Control Enforcement
# Integration test for the MRO simulator

def secure_analyze_report(role, report):
    authorization = authorize(role, "analyze_report")

    if not authorization["access_granted"]:
        return {
            "status": "ACCESS_DENIED",
            "role": role,
            "report_processed": False
        }

    analysis = analyze_report(report)

    return {
        "status": "SUCCESS",
        "role": role,
        "report_processed": True,
        "analysis": analysis
    }


# Integration test cases
test_cases = [
    ("maintenance_planner", True),
    ("quality_specialist", True),
    ("guest", False),
    ("auditor", False),
    ("unknown_role", False)
]

for role, expected in test_cases:
    response = secure_analyze_report(role, report)
    actual = response["report_processed"]

    status = "PASS" if actual == expected else "FAIL"

    print(
        f"{status} | {role} | "
        f"{response['status']} | "
        f"Processed: {actual}"
    )

    assert actual == expected, f"Test failed: {role}"


PASS | maintenance_planner | SUCCESS | Processed: True
PASS | quality_specialist | SUCCESS | Processed: True
PASS | guest | ACCESS_DENIED | Processed: False
PASS | auditor | ACCESS_DENIED | Processed: False
PASS | unknown_role | ACCESS_DENIED | Processed: False


In [ ]:

# CTRL-05 – Audit Logging
# Demonstration of traceable authorization events

from datetime import datetime, timezone
import uuid
import json

audit_log = []

def logged_analyze_report(role, report):
    event_id = str(uuid.uuid4())
    timestamp = datetime.now(timezone.utc).isoformat()

    response = secure_analyze_report(role, report)

    event = {
        "event_id": event_id,
        "timestamp": timestamp,
        "actor_role": role,
        "action": "analyze_report",
        "report_id": report["report_id"],
        "authorization": response["status"],
        "report_processed": response["report_processed"]
    }

    audit_log.append(event)
    return response


# TEST-05 – Audit trail reconstruction
roles = [
    "maintenance_planner",
    "guest",
    "auditor",
    "quality_specialist"
]

for role in roles:
    logged_analyze_report(role, report)

print("AUDIT LOG:")
print(json.dumps(audit_log, indent=2))

assert len(audit_log) == 4
assert sum(e["report_processed"] for e in audit_log) == 2

print("\nTEST-05: PASS – Four events recorded")


AUDIT LOG:
[
  {
    "event_id": "9ec838ea-2fbc-44c0-a444-7c1499c386ba",
    "timestamp": "2026-10-15T09:47:15.861047+00:00",
    "actor_role": "maintenance_planner",
    "action": "analyze_report",
    "report_id": "MRO-TEST-001",
    "authorization": "SUCCESS",
    "report_processed": true
  },
  {
    "event_id": "d79cc286-bb04-461d-9e16-48b2db3653b8",
    "timestamp": "2026-10-15T09:47:15.861127+00:00",
    "actor_role": "guest",
    "action": "analyze_report",
    "report_id": "MRO-TEST-001",
    "authorization": "ACCESS_DENIED",
    "report_processed": false
  },
  {
    "event_id": "bea25fe9-495d-4a1a-92a3-290dee708486",
    "timestamp": "2026-10-15T09:47:15.861156+00:00",
    "actor_role": "auditor",
    "action": "analyze_report",
    "report_id": "MRO-TEST-001",
    "authorization": "ACCESS_DENIED",
    "report_processed": false
  },
  {
    "event_id": "4fade8cc-b331-4ac3-b9e3-34a5262ab091",
    "timestamp": "2026-10-15T09:47:15.861180+00:00",
    "actor_role": "quality_spec

In [10]:

# CTRL-07 – Human Oversight
# TEST-06 – Human approval enforcement

from datetime import datetime, timezone
import uuid

approval_log = []

def request_human_approval(report_id):
    return {
        "approval_id": str(uuid.uuid4()),
        "report_id": report_id,
        "status": "PENDING",
        "approved_by": None,
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

def approve_request(request, reviewer_role, decision):
    if reviewer_role != "authorized_reviewer":
        return False

    if request["status"] != "PENDING":
        return False

    if decision not in ("APPROVED", "REJECTED"):
        return False

    request["status"] = decision
    request["approved_by"] = reviewer_role

    approval_log.append({
        "approval_id": request["approval_id"],
        "decision": decision,
        "reviewer_role": reviewer_role,
        "timestamp": datetime.now(timezone.utc).isoformat()
    })

    return True

def execute_action(request):
    return request["status"] == "APPROVED"


# TEST-06
request = request_human_approval("MRO-TEST-001")

assert execute_action(request) is False
print("PASS – Action blocked before approval")

assert approve_request(request, "guest", "APPROVED") is False
assert execute_action(request) is False
print("PASS – Unauthorized approval rejected")

assert approve_request(
    request, "authorized_reviewer", "APPROVED"
) is True

assert execute_action(request) is True
print("PASS – Authorized approval accepted")

assert len(approval_log) == 1
print("PASS – Approval decision recorded")

print("\nTEST-06: PASS – Human oversight controls")


PASS – Action blocked before approval
PASS – Unauthorized approval rejected
PASS – Authorized approval accepted
PASS – Approval decision recorded

TEST-06: PASS – Human oversight controls


In [11]:

# TEST-07: Agent Identity & Delegation Controls
from datetime import datetime, timezone, timedelta
import uuid

delegation_log = []

def check_delegation(agent_id, action, delegation):
    now = datetime.now(timezone.utc)
    allowed = (
        agent_id == delegation["agent_id"]
        and action in delegation["allowed_actions"]
        and now < delegation["expires_at"]
        and delegation["active"]
    )

    delegation_log.append({
        "event_id": str(uuid.uuid4()),
        "agent_id": agent_id,
        "action": action,
        "decision": "ALLOW" if allowed else "DENY",
        "timestamp": now.isoformat()
    })
    return allowed

delegation = {
    "agent_id": "MRO-AGENT-001",
    "allowed_actions": ["analyze_report"],
    "expires_at": datetime.now(timezone.utc) + timedelta(hours=1),
    "active": True
}

assert check_delegation(
    "MRO-AGENT-001", "analyze_report", delegation
) is True
print("PASS - Authorized agent")

assert check_delegation(
    "UNKNOWN-AGENT", "analyze_report", delegation
) is False
print("PASS - Unknown agent blocked")

assert check_delegation(
    "MRO-AGENT-001", "approve_maintenance", delegation
) is False
print("PASS - Unauthorized action blocked")

expired = {**delegation,
           "expires_at": datetime.now(timezone.utc) - timedelta(hours=1)}

assert check_delegation(
    "MRO-AGENT-001", "analyze_report", expired
) is False
print("PASS - Expired delegation blocked")

print("\nTEST-07: PASS - Agent identity and delegation controls")
print("Audit events:", len(delegation_log))


PASS - Authorized agent
PASS - Unknown agent blocked
PASS - Unauthorized action blocked
PASS - Expired delegation blocked

TEST-07: PASS - Agent identity and delegation controls
Audit events: 4


In [1]:

import json
import hashlib
from copy import deepcopy
from datetime import datetime, timezone

# TEST-08: Audit Trail Integrity
# Synthetic data - deterministic proof of concept

def canonical_json(data):
    return json.dumps(
        data, sort_keys=True, separators=(",", ":")
    )

def calculate_hash(data):
    return hashlib.sha256(
        canonical_json(data).encode("utf-8")
    ).hexdigest()

audit_chain = []

def add_audit_event(event_id, actor, action, decision):
    previous_hash = (
        audit_chain[-1]["hash"] if audit_chain else "GENESIS"
    )

    event = {
        "event_id": event_id,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "actor": actor,
        "action": action,
        "decision": decision,
        "previous_hash": previous_hash
    }

    event["hash"] = calculate_hash(event)
    audit_chain.append(event)

def verify_audit_chain(chain):
    previous_hash = "GENESIS"

    for event in chain:
        if event["previous_hash"] != previous_hash:
            return False

        content = {
            k: v for k, v in event.items() if k != "hash"
        }

        if calculate_hash(content) != event["hash"]:
            return False

        previous_hash = event["hash"]

    return True

# Create synthetic audit events
add_audit_event("EVT-001", "agent-001", "analyze_report", "ALLOW")
add_audit_event("EVT-002", "guest", "approve_action", "DENY")
add_audit_event("EVT-003", "reviewer-001", "approve_action", "ALLOW")

# Scenario 1: Original chain
assert verify_audit_chain(audit_chain) is True
print("PASS - Original audit chain verified")

# Scenario 2: Modified decision
tampered = deepcopy(audit_chain)
tampered[1]["decision"] = "ALLOW"

assert verify_audit_chain(tampered) is False
print("PASS - Unauthorized modification detected")

# Scenario 3: Deleted middle event
deleted = deepcopy(audit_chain)
del deleted[1]

assert verify_audit_chain(deleted) is False
print("PASS - Deleted middle event detected")

# Scenario 4: Reordered events
reordered = deepcopy(audit_chain)
reordered[0], reordered[1] = reordered[1], reordered[0]

assert verify_audit_chain(reordered) is False
print("PASS - Event reordering detected")

print("\nTEST-08: PASS - Audit trail integrity controls")
print("Audit events:", len(audit_chain))


PASS - Original audit chain verified
PASS - Unauthorized modification detected
PASS - Deleted middle event detected
PASS - Event reordering detected

TEST-08: PASS - Audit trail integrity controls
Audit events: 3
